# EasyRAG Prompting and answer style

## Chapter position

By this point the evidence is already selected and packed. The only question left is how the prompt frames that evidence for the answer stage.

## Learning question

What changes when you switch `AnswerParam.style`, `allow_abstain`, or `require_citations`, and what stays fixed?

## Success criteria

- You can explain how each answer style changes the instruction layer.
- You can show that the evidence block stays the same across styles.
- You can explain how `generate_answer()` passes style and citations to a custom answer model.

## Source code anchors

- `easyrag.rag.generation.prompting.build_generation_prompt`
- `easyrag.rag.types.AnswerParam`
- `easyrag.rag.generation.pipeline.generate_answer`

## Method principles

- `easyrag.rag.generation.prompting.build_generation_prompt`: This prompt builder combines answer instructions with the packed evidence block. It is where style, citation requirements, and abstention rules become model-facing text.
- `easyrag.rag.types.AnswerParam`: This dataclass is the answer-side policy bundle. It controls citation budget, context budget, answer style, and abstention rules without changing retrieval behavior.
- `easyrag.rag.generation.pipeline.generate_answer`: This is the answer orchestration wrapper. It reruns citation selection, packs context, builds the prompt, calls synthesis, and returns a structured `AnswerResult`.

Design reason: these anchors break answering into evidence selection, packing, prompting, and synthesis so the notebook can show that answer quality is a data-shaping problem as much as a model-wording problem.


## How the code fits together

`build_generation_prompt()` is a small function, but it controls a real boundary. `AnswerParam.style` changes the wording of the instruction layer, while `allow_abstain` and `require_citations` toggle extra guardrail lines. The packed evidence block does not change when those flags change. Later, `generate_answer()` passes both the final prompt and the explicit `style` and `citations` kwargs to the answer model, so a custom model can react to configuration without reparsing the whole prompt.

Design reason: the notebook walks the same evidence through smaller answer-stage boundaries before it asks for a final sentence. That ordering is what lets you see whether a failure came from evidence budget, context packing, prompting policy, or synthesis.


## What this cell is proving

We keep the context block fixed so the only moving part is the instruction layer. That makes prompt differences easy to trust.

In [ ]:
import importlib
import sys
import tempfile
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "notebooks" / "_utils.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

_notebook_utils = importlib.import_module("notebooks._utils")
_rag = importlib.import_module("easyrag.rag")
_packing = importlib.import_module("easyrag.rag.generation.packing")
_pipeline = importlib.import_module("easyrag.rag.generation.pipeline")
_prompting = importlib.import_module("easyrag.rag.generation.prompting")
_async_utils = importlib.import_module("easyrag.support.async_utils")

ensure_repo_root_on_path = _notebook_utils.ensure_repo_root_on_path
_print_json = _notebook_utils.print_json
can_use_openai_compatible_models = _notebook_utils.provider_ready
skip_message = _notebook_utils.skip_message
_stub_embedding = _notebook_utils.stub_embedding
_stub_query_model = _notebook_utils.stub_query_model
AnswerParam = _rag.AnswerParam
EasyRAG = _rag.EasyRAG
QueryParam = _rag.QueryParam
build_context_block = _packing.build_context_block
generate_answer = _pipeline.generate_answer
build_generation_prompt = _prompting.build_generation_prompt
run_sync = _async_utils.run_sync

REPO_ROOT = ensure_repo_root_on_path()

## What this cell is proving

Style changes should show up in the prompt header first. The evidence block should stay untouched.

Design reason: generation is intentionally split into evidence selection, packing, prompting, and synthesis because each boundary can fail for a different reason. Keeping this step explicit is what makes answer quality debuggable instead of treating the model response as one indivisible artifact.


In [ ]:
question = "How does EasyRAG keep answers grounded?"
selected_citations = [
    {
        "title": "Retrieval",
        "location": "docs/retrieval.md",
        "snippet": "Retrieval keeps query rewrite and grounded evidence visible.",
    },
    {
        "title": "Policy",
        "location": "docs/policy.md",
        "snippet": "Citation-aware answers expose the evidence budget to readers.",
    },
]
context_block = build_context_block(selected_citations)
styles = {
    "citation_aware": AnswerParam(
        style="citation_aware", require_citations=True, allow_abstain=True
    ),
    "extractive": AnswerParam(
        style="extractive", require_citations=True, allow_abstain=True
    ),
    "abstractive": AnswerParam(
        style="abstractive", require_citations=True, allow_abstain=True
    ),
}
prompts = {
    label: build_generation_prompt(question, context_block, param)
    for label, param in styles.items()
}

print("Shared evidence block")
print("-" * 40)
print(context_block)
print()
for label, prompt in prompts.items():
    print(f"=== {label} ===")
    print("\n".join(prompt.splitlines()[:8]))
    print()

## Why this output looks like this

The first few lines are where style lives. `citation_aware` asks for concise grounded answering, `extractive` pushes the wording closer to the evidence, and `abstractive` gives the model a little more freedom while still insisting on grounding. The evidence block below those instructions is identical.

Design reason: the output looks layered because the same evidence is being carried through several answer-stage representations. Keeping the citation subset, packed context, prompt text, and final answer close together is what exposes whether a failure came from selection, packing, instruction design, or synthesis.


## What this cell is proving

Style is not only prompt text. It also reaches the synthesis boundary as an explicit argument. That matters when you plug in a custom answer model.

Design reason: generation is intentionally split into evidence selection, packing, prompting, and synthesis because each boundary can fail for a different reason. Keeping this step explicit is what makes answer quality debuggable instead of treating the model response as one indivisible artifact.


In [ ]:
def style_echo_answer_model(prompt: str, **kwargs):
    return f"style={kwargs.get('style')} | citations={len(kwargs.get('citations', []))}"


prompt_tmp = tempfile.TemporaryDirectory()
prompt_rag = EasyRAG(
    working_dir=prompt_tmp.name,
    workspace="prompt-style-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
    answer_model_func=style_echo_answer_model,
)
run_sync(prompt_rag.initialize_storages())
run_sync(
    prompt_rag.ainsert(
        "# Retrieval\nEasyRAG keeps answer synthesis grounded in citations.\n",
        ids=["doc::retrieval"],
        file_paths=["docs/retrieval.md"],
    )
)
retrieval_result = run_sync(
    prompt_rag.aquery(
        question,
        QueryParam(
            mode="naive", rewrite_enabled=False, mqe_enabled=False, chunk_top_k=3
        ),
    )
)

style_outputs = {}
for label, param in styles.items():
    answer = generate_answer(
        question,
        retrieval_result,
        answer_param=param,
        answer_model_func=prompt_rag.answer_model_func,
    )
    style_outputs[label] = {
        "prompt_preview": answer.prompt.splitlines()[:8],
        "answer": answer.answer,
        "metadata": {
            "style": answer.metadata["style"],
            "answer_model_used": answer.metadata["answer_model_used"],
            "fallback_used": answer.metadata["fallback_used"],
        },
    }

_print_json(style_outputs)

## Why this output looks like this

The echoed answer proves that `style` and `citations` arrive as explicit kwargs. The prompt preview still matters, but the answer model does not have to recover configuration by scraping the prompt text.

Design reason: the output is intentionally small and legible because it is establishing the reference state for the rest of the notebook. The next stage reuses exactly these shaped inputs so you can tell which later fields are preserved and which ones are newly derived.


## What this cell is proving

The evidence tail of the prompt is stable across styles. If a prompt bug only shows up in one style, it is probably in the instruction layer, not in context packing.

Design reason: the notebook uses hand-shaped inputs here so the later behavior can be attributed to the stage under study instead of to noisy source material. When the examples have obvious roles and boundaries, it becomes much easier to tell whether the pipeline preserved the intended structure.


In [ ]:
evidence_tails = {
    label: prompt.split("Retrieved evidence:\n", 1)[1]
    for label, prompt in prompts.items()
}
_print_json(
    {
        "tails_match": {
            "citation_vs_extractive": evidence_tails["citation_aware"]
            == evidence_tails["extractive"],
            "citation_vs_abstractive": evidence_tails["citation_aware"]
            == evidence_tails["abstractive"],
        },
        "evidence_tail": evidence_tails["citation_aware"],
    }
)

## What this cell is proving

The provider-backed path should preserve the same prompt contract even when the answer text becomes less deterministic.

In [ ]:
if not can_use_openai_compatible_models():
    print(skip_message("provider"))
else:
    provider_param = AnswerParam(style="citation_aware", require_citations=True)
    provider_prompt = build_generation_prompt(question, context_block, provider_param)
    print("\n".join(provider_prompt.splitlines()[:10]))

## Common mistakes / debugging cues

- If two styles produce the same prompt header, check the `AnswerParam` values first. The bug may be configuration, not generation.
- Do not blame citation formatting on style alone. `require_citations` is a separate switch.
- If a custom answer model behaves differently from the built-in fallback path, inspect both the prompt and the explicit kwargs it received.

In [ ]:
run_sync(prompt_rag.finalize_storages())
prompt_tmp.cleanup()
print("Cleaned up the prompting workspace.")

## Takeaway

Answer style lives in the instruction layer. The evidence block stays fixed. That split is what makes prompt debugging tractable.

Continue with [05_05_generation_failures_and_guardrails.ipynb](05_05_generation_failures_and_guardrails.ipynb) to see how abstention, fallback, and citation requirements change failure behavior.